In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LambdaCallback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from tqdm import tqdm

ImportError: cannot import name 'keras' from 'tensorflow' (unknown location)

In [9]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("TF location:", tf.__file__)


AttributeError: module 'tensorflow' has no attribute '__version__'

In [ ]:
### Data Loading
# Original data
train_mask_folder = "Train-Mask"
train_input_folder = "Train-Input"
test_input_folder = "Test-Input"
test_mask_folder = "Test-Mask"

# Augmented data
# aug_input_folder = "output6-input"
# aug_mask_folder = "output6-mask"

def dice_acc(y_true, y_pred, smooth=1e-7):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])
    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth), axis=0)

def dice_loss(y_true, y_pred):
    return 1 - dice_acc(y_true, y_pred)

### U-Net Model Definition
def unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)
    
    # Encoder
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    
    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)
    
    # Decoder
    u6 = layers.Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)
    
    u7 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)
    
    u8 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)
    
    u9 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)
    
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model



def load_images(folder, is_mask=False):
    images = []
    for filename in os.listdir(folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = resize(img, (256, 256), mode='constant', preserve_range=True)
            if is_mask:
                img = (img > 127).astype(np.float32)  # Binarize masks
            images.append(img)
    return np.array(images)

# Load original
Train_Inputs = load_images(train_input_folder)
Train_Masks = load_images(train_mask_folder, is_mask=True)
Test_Inputs = load_images(test_input_folder)
Test_Masks = load_images(test_mask_folder, is_mask=True)

# # Load augmented
# Aug_Inputs = load_images(aug_input_folder)
# Aug_Masks = load_images(aug_mask_folder, is_mask=True)

# Expand dims for channel (grayscale)
Train_Inputs = np.expand_dims(Train_Inputs, axis=-1)
Train_Masks = np.expand_dims(Train_Masks, axis=-1)
Test_Inputs = np.expand_dims(Test_Inputs, axis=-1)
Test_Masks = np.expand_dims(Test_Masks, axis=-1)
# Aug_Inputs = np.expand_dims(Aug_Inputs, axis=-1)
# Aug_Masks = np.expand_dims(Aug_Masks, axis=-1)

print(f"Original Training Inputs: {Train_Inputs.shape}, Masks: {Train_Masks.shape}")
# print(f"Augmented Inputs: {Aug_Inputs.shape}, Masks: {Aug_Masks.shape}")

### Custom Callback for Epoch Logging
epoch_callback = LambdaCallback(
    on_epoch_end=lambda epoch, logs: print(f"Epoch {epoch + 1}: Loss = {logs['loss']:.4f}, Dice Acc = {logs['dice_acc']:.4f}, Val Loss = {logs['val_loss']:.4f}, Val Dice Acc = {logs['val_dice_acc']:.4f}")
)

### Baseline Training (Original Data Only)
print("Training Baseline Model on Original Data...")
model_baseline = unet(input_shape=(256, 256, 1))
model_baseline.compile(optimizer=Adam(learning_rate=1e-4), loss=dice_loss, metrics=[dice_acc])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

results_baseline = model_baseline.fit(Train_Inputs, Train_Masks,
                                     batch_size=8,
                                     epochs=50,
                                     validation_data=(Test_Inputs, Test_Masks),
                                     callbacks=[early_stop, reduce_lr, epoch_callback],
                                     verbose=1)

# Save baseline results
baseline_val_dice = max(results_baseline.history['val_dice_acc'])
print(f"\nBaseline Max Validation Dice Score: {baseline_val_dice:.4f}")

### Combined Training (Original + Augmented)
print("\nTraining Model on Combined Data...")
